# Jalon B2 - Metrics & Attribution Demo

Notebook de demonstration B2: protocole metrics, attribution layer x cell, et sauvegarde d'artefacts.

In [ ]:
import tempfile
import torch
from torch.utils.data import DataLoader, TensorDataset

from boardGPT.nn import GPT, GPTConfig
from boardGPT.interpretability import (
    LinearBoardProbe,
    ProtocolConfig,
    run_b2_metrics_protocol,
    run_b2_attribution_protocol,
    compute_intervention_attribution_summary,
    save_run_artifacts,
    load_run_artifacts,
)

In [ ]:
cfg = GPTConfig(block_size=8, vocab_size=25, n_layer=2, n_head=1, n_embd=16, dropout=0.0, bias=True)
model = GPT(cfg, use_flash=False)

probe = LinearBoardProbe(input_dim=16)
with torch.no_grad():
    probe.proj.weight.zero_()
    probe.proj.bias.zero_()
    base = 5 * 3
    probe.proj.weight[base + 0, 0] = -2.0
    probe.proj.weight[base + 1, 0] = -1.0
    probe.proj.weight[base + 2, 0] = 2.0

In [ ]:
tokens = torch.randint(0, cfg.vocab_size, (12, cfg.block_size))
loader = DataLoader(TensorDataset(tokens), batch_size=3, shuffle=False)
pconf = ProtocolConfig(max_examples=6, intervention_steps=20, intervention_lr=0.08, intervention_reg_strength=1e-4, device='cpu')

metrics_report = run_b2_metrics_protocol(
    model=model,
    probe=probe,
    batches=loader,
    layers=[0, 1],
    target_cell=5,
    target_class=2,
    config=pconf,
)
metrics_report['metrics']

In [ ]:
loader2 = DataLoader(TensorDataset(tokens), batch_size=3, shuffle=False)
attr_report = run_b2_attribution_protocol(
    model=model,
    probe=probe,
    batches=loader2,
    layers=[0, 1],
    target_cells=[3, 5, 9],
    target_class=2,
    config=pconf,
    top_k=6,
)
attr_map = attr_report['attribution']['attribution_map']
print('attribution map shape:', tuple(attr_map.shape))
attr_report['attribution']['top_contributors']

In [ ]:
# Resume attribution summary from one subset
subset_results = attr_report['detailed_reports'][0][5]['results']
summary = compute_intervention_attribution_summary({0: subset_results}, top_k=3)
summary['top_contributors']

In [ ]:
with tempfile.TemporaryDirectory() as tmp_dir:
    _ = save_run_artifacts(
        run_dir=tmp_dir,
        config={'stage': 'B2-demo'},
        summary=metrics_report['metrics']['overall'],
        results=metrics_report['sweep']['layer_results'][0]['results'],
        tensors={'attribution_map': attr_map},
    )
    loaded = load_run_artifacts(tmp_dir)
    print('loaded config:', loaded['config'])
    print('tensor keys:', list(loaded['tensors'].keys()))